# 02 – Feature Engineering

**Objective:** Construct economically meaningful features from the cleaned datasets that will power clustering, classification, and forecasting models.

**Features to build:**
1. **Export / Import Growth Rate** – year-over-year percentage change in trade values.
2. **Global Demand Index** – normalised total import demand per product sector & year.
3. **Market Penetration Ratio** – Algeria’s export share relative to target-market size (proxy via GDP or imports).
4. **Trade Balance Indicators** – export minus import values; trade balance as % of GDP.
5. **Product Diversification Metrics** – Herfindahl–Hirschman Index (HHI) and entropy-based diversification scores.
6. **Economic Context Features** – GDP per capita, trade openness, population.

**Input:** `data/processed/01_*.csv`
**Output:** `data/processed/02_features_*.csv`

## 2.1 – Setup

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# Load cleaned datasets
df_comtrade = pd.read_csv(PROCESSED_DIR / '01_comtrade_cleaned.csv')
df_wb = pd.read_csv(PROCESSED_DIR / '01_worldbank_cleaned.csv')
df_wto = pd.read_csv(PROCESSED_DIR / '01_wto_cleaned.csv')

print('Comtrade shape:', df_comtrade.shape)
print('World Bank shape:', df_wb.shape)
print('WTO shape:', df_wto.shape)

Comtrade shape: (6, 14)
World Bank shape: (5375, 9)
WTO shape: (450, 10)


## 2.2 – Trade Growth Rates (Comtrade)

Compute year-over-year growth for Algeria’s exports and imports.
When product-level detail is available, growth is computed per product; otherwise at the aggregate level.

In [2]:
# Ensure numeric types
df_comtrade['refyear'] = pd.to_numeric(df_comtrade['refyear'], errors='coerce')
for col in ['fobvalue', 'cifvalue', 'primaryvalue', 'netwgt']:
    if col in df_comtrade.columns:
        df_comtrade[col] = pd.to_numeric(df_comtrade[col], errors='coerce')

# Choose the best monetary column available
value_col = None
for col in ['primaryvalue', 'fobvalue', 'cifvalue']:
    if col in df_comtrade.columns and df_comtrade[col].notna().any():
        value_col = col
        break
print('Using monetary column:', value_col)

def compute_growth(df, group_cols, value_col, new_col_name='growth_rate_pct'):
    """
    Compute year-over-year percentage growth for a given value column.
    Handles zero/negative previous values gracefully.
    """
    df = df.sort_values(by=group_cols + ['refyear']).copy()
    df[new_col_name] = df.groupby(group_cols)[value_col].pct_change() * 100
    # For first year or when prev value is 0/NaN, set to NaN (to be imputed later)
    return df

if value_col:
    # Growth at the flow_type + cmdcode level (product or total)
    group_cols = ['flow_type']
    if 'cmdcode' in df_comtrade.columns and df_comtrade['cmdcode'].nunique() > 1:
        group_cols.append('cmdcode')
    df_comtrade = compute_growth(df_comtrade, group_cols, value_col, 'trade_growth_rate_pct')

    # Absolute change too
    df_comtrade = df_comtrade.sort_values(by=group_cols + ['refyear']).copy()
    df_comtrade['trade_abs_change'] = df_comtrade.groupby(group_cols)[value_col].diff()

print('Growth stats:')
print(df_comtrade[['refyear', 'flow_type', value_col, 'trade_growth_rate_pct', 'trade_abs_change']].head(10)) if value_col else print('No value column available')

Using monetary column: primaryvalue
Growth stats:
   refyear flow_type  primaryvalue  trade_growth_rate_pct  trade_abs_change
0       52    export             0                    NaN               NaN
1       52    export             0                    NaN               0.0
2       52    export             0                    NaN               0.0
3       52    import             0                    NaN               NaN
4       52    import             0                    NaN               0.0
5       52    import             0                    NaN               0.0


## 2.3 – Trade Balance Indicators

Trade balance = Exports – Imports.
When detailed partner/product info exists, compute at that level; otherwise compute at aggregate level.

In [3]:
if value_col:
    # Pivot exports / imports at the most granular level possible
    pivot_index = ['refyear']
    if 'cmdcode' in df_comtrade.columns:
        pivot_index.append('cmdcode')
    if 'partneriso' in df_comtrade.columns:
        pivot_index.append('partneriso')

    # Aggregate duplicates just in case
    df_trade_bal = df_comtrade.groupby(pivot_index + ['flow_type'])[value_col].sum().reset_index()
    df_trade_bal_pivot = df_trade_bal.pivot(index=pivot_index, columns='flow_type', values=value_col).reset_index()

    # Normalise column names
    col_map = {}
    for c in df_trade_bal_pivot.columns:
        s = str(c).lower()
        if 'export' in s:
            col_map[c] = 'exports'
        elif 'import' in s:
            col_map[c] = 'imports'
    df_trade_bal_pivot.rename(columns=col_map, inplace=True)

    if 'exports' in df_trade_bal_pivot.columns and 'imports' in df_trade_bal_pivot.columns:
        df_trade_bal_pivot['trade_balance'] = df_trade_bal_pivot['exports'].fillna(0) - df_trade_bal_pivot['imports'].fillna(0)
        df_trade_bal_pivot['trade_balance_pct_of_exports'] = (
            df_trade_bal_pivot['trade_balance'] / df_trade_bal_pivot['exports'].replace(0, np.nan) * 100
        )
        print('Trade balance computed. Sample:')
        display(df_trade_bal_pivot.head())
    else:
        print('Could not compute trade balance: missing export or import column after pivot.')
else:
    print('Skipping trade balance (no value column).')
    df_trade_bal_pivot = pd.DataFrame()

Trade balance computed. Sample:


flow_type,refyear,cmdcode,partneriso,exports,imports,trade_balance,trade_balance_pct_of_exports
0,52,All Commodities,World,0,0,0,NaN


## 2.4 – Global Demand Index (WTO)

For each product sector, compute:
- **Total global demand** = sum of import values (or indicator values) per year.
- **Demand index** = sector value / max sector value in that year (0–1 scale).
- **Demand growth** = year-over-year % change in sector demand.

In [4]:
# Ensure numeric
df_wto['year'] = pd.to_numeric(df_wto['year'], errors='coerce')
df_wto['value'] = pd.to_numeric(df_wto['value'], errors='coerce')

# Use 'indicator' to separate exports vs imports if present
if 'indicator' in df_wto.columns:
    print('WTO indicators:')
    print(df_wto['indicator'].value_counts().head(10))

# Compute yearly totals per product_sector (treating value as demand proxy)
if 'product_sector' in df_wto.columns:
    # Aggregate by year + product_sector
    wto_agg = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()

    # Global demand index per year (max-normalised)
    yearly_max = wto_agg.groupby('year')['value'].transform('max')
    wto_agg['global_demand_index'] = wto_agg['value'] / yearly_max.replace(0, np.nan)

    # Year-over-year demand growth per sector
    wto_agg = wto_agg.sort_values(['product_sector', 'year']).copy()
    wto_agg['demand_growth_rate_pct'] = wto_agg.groupby('product_sector')['value'].pct_change() * 100

    # Rolling 3-year average demand
    wto_agg['demand_3yr_ma'] = wto_agg.groupby('product_sector')['value'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

    print('WTO demand features computed. Sample:')
    display(wto_agg.head(10))
else:
    wto_agg = pd.DataFrame()
    print('No product_sector column found in WTO data.')

WTO indicators:
indicator
Merchandise Imports By Product Group � Annual    181
Merchandise Exports By Product Group � Annual    181
Total Merchandise Imports - Quarterly             44
Total Merchandise Exports - Quarterly             44
Name: count, dtype: int64
WTO demand features computed. Sample:


,year,product_sector,value,global_demand_index,demand_growth_rate_pct,demand_3yr_ma
0,2015,Agricultural Products,11007.890104,0.063725,NaN,11007.890104
18,2016,Agricultural Products,9971.365942,0.064653,-9.416193,10489.628023
36,2017,Agricultural Products,10176.748441,0.062626,2.059723,10385.334829
54,2018,Agricultural Products,8674.170198,0.049214,-14.764817,9607.428194
72,2019,Agricultural Products,9145.108000,0.057193,5.429197,9332.008880
90,2020,Agricultural Products,9326.060000,0.081126,1.978675,9048.446066
108,2021,Agricultural Products,10786.671263,0.070872,15.661611,9752.613088
126,2022,Agricultural Products,12834.879324,0.061362,18.988324,10982.536862
144,2023,Agricultural Products,11793.402072,0.059928,-8.114430,11804.984220
162,2024,Agricultural Products,11924.018211,0.063156,1.107536,12184.099869


## 2.5 – Market Penetration Ratio (World Bank proxy)

Since granular bilateral export data is limited, we proxy market penetration as:

`market_penetration_proxy = (Algeria's exports to World) / (Target country's GDP)`

This tells us, in relative terms, how large Algeria’s export footprint is versus the economic size of potential destination markets.

For countries with high import ratios, we also compute `export_to_import_ratio_proxy`.

In [5]:
# Prepare Algeria aggregate exports by year from Comtrade
if value_col and not df_comtrade.empty:
    algeria_exports = (
        df_comtrade[df_comtrade['flow_type'].str.lower() == 'export']
        .groupby('refyear')[value_col]
        .sum()
        .reset_index()
        .rename(columns={value_col: 'algeria_exports_world'})
    )
    print('Algeria exports by year:')
    display(algeria_exports)
else:
    algeria_exports = pd.DataFrame()
    print('No Comtrade export data available for penetration ratio.')

# Merge with World Bank GDP data
if not algeria_exports.empty and not df_wb.empty:
    # Ensure year columns are int
    df_wb['year'] = pd.to_numeric(df_wb['year'], errors='coerce')
    gdp_col = 'NY.GDP.MKTP.CD'
    if gdp_col in df_wb.columns:
        df_pen = df_wb[['country', 'country_code', 'year', gdp_col]].copy()
        df_pen = df_pen.merge(algeria_exports, left_on='year', right_on='refyear', how='left')
        df_pen['market_penetration_proxy'] = df_pen['algeria_exports_world'] / df_pen[gdp_col].replace(0, np.nan) * 100
        print('Market penetration proxy computed (Algeria exports / country GDP * 100):')
        display(df_pen.dropna(subset=['market_penetration_proxy']).head(10))
    else:
        print(f'GDP column {gdp_col} not found in World Bank data.')
else:
    print('Skipping market penetration (missing required inputs).')

Algeria exports by year:


,refyear,algeria_exports_world
0,52,0


Market penetration proxy computed (Algeria exports / country GDP * 100):


,country,country_code,year,NY.GDP.MKTP.CD,refyear,algeria_exports_world,market_penetration_proxy


## 2.6 – Product Diversification Metrics (WTO)

Using the Herfindahl–Hirschman Index (HHI) and Shannon Entropy on Algeria’s trade by product sector.

- **HHI** = sum of squared shares (0 = perfectly diversified, 1 = perfectly concentrated).
- **Shannon Entropy** = -sum(share * ln(share)) (higher = more diversified).

In [6]:
if not df_wto.empty and 'product_sector' in df_wto.columns:
    # Yearly sector shares for Algeria
    wto_yearly = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()
    yearly_totals = wto_yearly.groupby('year')['value'].transform('sum')
    wto_yearly['share'] = wto_yearly['value'] / yearly_totals.replace(0, np.nan)

    # HHI per year
    hhi = wto_yearly.groupby('year').apply(
        lambda x: (x['share'] ** 2).sum()
    ).reset_index(name='hhi')

    # Shannon entropy per year
    def shannon_entropy(shares):
        shares = shares[shares > 0]
        return -(shares * np.log(shares)).sum()

    entropy = wto_yearly.groupby('year').apply(
        lambda x: shannon_entropy(x['share'])
    ).reset_index(name='shannon_entropy')

    diversification = hhi.merge(entropy, on='year')
    # Normalise entropy to [0,1] for comparability
    max_entropy = np.log(wto_yearly['product_sector'].nunique()) if wto_yearly['product_sector'].nunique() > 0 else 1
    diversification['shannon_entropy_norm'] = diversification['shannon_entropy'] / max_entropy

    print('Diversification metrics per year:')
    display(diversification)
else:
    diversification = pd.DataFrame()
    print('Skipping diversification (no product_sector data).')

Diversification metrics per year:


,year,hhi,shannon_entropy,shannon_entropy_norm
0,2015,0.275791,1.811936,0.626887
1,2016,0.278157,1.806677,0.625067
2,2017,0.279159,1.792764,0.620254
3,2018,0.291695,1.733755,0.599838
4,2019,0.287931,1.764823,0.610587
5,2020,0.291371,1.775879,0.614412
6,2021,0.293458,1.733184,0.599640
7,2022,0.302313,1.643553,0.568630
8,2023,0.292247,1.710958,0.591951
9,2024,0.287524,1.741922,0.602664


## 2.7 – Economic Context Features (World Bank)

Derive additional macro indicators:
- **Trade Openness** = Merchandise trade (% of GDP) already available, plus we compute `Trade / GDP` in absolute terms where both series exist.
- **GDP per Capita** already available.
- **Import Intensity** = Imports of goods & services (% of GDP).
- **GDP Growth** lagged by 1 year (proxy for economic momentum).

In [7]:
# Create explicit numeric copies
df_wb['year'] = pd.to_numeric(df_wb['year'], errors='coerce')

# Ensure indicator columns are numeric
indicator_cols = ['NY.GDP.MKTP.CD', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD',
                  'SP.POP.TOTL', 'TG.VAL.TOTL.GD.ZS', 'NE.IMP.GNFS.ZS']
for col in indicator_cols:
    if col in df_wb.columns:
        df_wb[col] = pd.to_numeric(df_wb[col], errors='coerce')

# Lagged GDP growth (1 year)
growth_col = 'NY.GDP.MKTP.KD.ZG'
if growth_col in df_wb.columns:
    df_wb = df_wb.sort_values(['country_code', 'year']).copy()
    df_wb['gdp_growth_lag1'] = df_wb.groupby('country_code')[growth_col].shift(1)

# Absolute trade volume proxy = trade_pct_gdp * gdp / 100
trade_col = 'TG.VAL.TOTL.GD.ZS'
gdp_col = 'NY.GDP.MKTP.CD'
if trade_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['trade_volume_proxy'] = df_wb[trade_col] * df_wb[gdp_col] / 100

# Absolute import volume proxy = import_pct_gdp * gdp / 100
imp_col = 'NE.IMP.GNFS.ZS'
if imp_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['import_volume_proxy'] = df_wb[imp_col] * df_wb[gdp_col] / 100

# Population-normalised GDP (already per capita exists, but just in case)
pop_col = 'SP.POP.TOTL'
if gdp_col in df_wb.columns and pop_col in df_wb.columns:
    df_wb['gdp_per_capita_computed'] = df_wb[gdp_col] / df_wb[pop_col].replace(0, np.nan)

print('Economic context features added. Sample:')
display(df_wb.head())
print('New columns:', [c for c in df_wb.columns if c not in indicator_cols + ['country', 'country_code', 'year']])

Economic context features added. Sample:


,country,country_code,year,NE.IMP.GNFS.ZS,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.CD,SP.POP.TOTL,TG.VAL.TOTL.GD.ZS,gdp_growth_lag1,trade_volume_proxy,import_volume_proxy,gdp_per_capita_computed
200,Aruba,ABW,2000.0,70.686869,1.873453e+09,7.622921,20681.023027,90588.0,272.544938,NaN,5.106000e+09,1.324285e+09,20681.023027
201,Aruba,ABW,2001.0,69.394325,1.896457e+09,4.182002,20740.132583,91439.0,252.839903,7.622921,4.795000e+09,1.316034e+09,20740.132583
202,Aruba,ABW,2002.0,68.666458,1.961844e+09,-0.944953,21307.248251,92074.0,178.658484,4.182002,3.505000e+09,1.347128e+09,21307.248251
203,Aruba,ABW,2003.0,70.063078,2.044112e+09,1.110505,21949.485996,93128.0,217.649551,-0.944953,4.449000e+09,1.432168e+09,21949.485996
204,Aruba,ABW,2004.0,67.765371,2.254831e+09,7.293728,23700.631990,95138.0,311.508971,1.110505,7.024000e+09,1.527994e+09,23700.631990


New columns: ['gdp_growth_lag1', 'trade_volume_proxy', 'import_volume_proxy', 'gdp_per_capita_computed']


## 2.8 – Save Feature-Engineered Datasets

In [8]:
# 1. Comtrade with growth & balance
df_comtrade.to_csv(PROCESSED_DIR / '02_comtrade_features.csv', index=False)
print('Saved: 02_comtrade_features.csv')

# 2. Trade balance (if computed)
if not df_trade_bal_pivot.empty:
    df_trade_bal_pivot.to_csv(PROCESSED_DIR / '02_trade_balance.csv', index=False)
    print('Saved: 02_trade_balance.csv')

# 3. WTO with demand index & diversification
if not wto_agg.empty:
    wto_agg.to_csv(PROCESSED_DIR / '02_wto_demand_features.csv', index=False)
    print('Saved: 02_wto_demand_features.csv')

if not diversification.empty:
    diversification.to_csv(PROCESSED_DIR / '02_diversification_metrics.csv', index=False)
    print('Saved: 02_diversification_metrics.csv')

# 4. World Bank with enriched macro features
df_wb.to_csv(PROCESSED_DIR / '02_worldbank_features.csv', index=False)
print('Saved: 02_worldbank_features.csv')

print('\nFeature engineering complete. Proceed to 03_scaling_and_normalization.ipynb.')

Saved: 02_comtrade_features.csv
Saved: 02_trade_balance.csv
Saved: 02_wto_demand_features.csv
Saved: 02_diversification_metrics.csv


Saved: 02_worldbank_features.csv



Feature engineering complete. Proceed to 03_scaling_and_normalization.ipynb.
